In [8]:
import pandas as pd
import re
import io
from collections import defaultdict

# Function to parse the log file
def parse_log_file(log_content):
    lines = log_content.split('\n')
    parsed_logs = []
    
    # Regular expressions to extract information
    timestamp_pattern = re.compile(r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})')
    log_level_pattern = re.compile(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3} \[(INFO|WARNING|ERROR|DEBUG)\]')
    group_pattern = re.compile(r'Group: (\d+)')
    user_pattern = re.compile(r'User: (\d+)')
    
    last_timestamp = None
    last_log_level = None
    last_group_id = None
    last_user_id = None
    
    for line in lines:
        if not line.strip():
            continue
            
        # Extract timestamp
        timestamp_match = timestamp_pattern.search(line)
        if timestamp_match:
            last_timestamp = timestamp_match.group(1)
        
        # Extract log level
        log_level_match = log_level_pattern.search(line)
        if log_level_match:
            last_log_level = log_level_match.group(1)
        
        # Extract group ID
        group_match = group_pattern.search(line)
        if group_match:
            last_group_id = group_match.group(1)
        
        # Extract user ID
        user_match = user_pattern.search(line)
        if user_match:
            last_user_id = user_match.group(1)
        
        # Add the parsed log entry
        parsed_logs.append({
            'timestamp': last_timestamp,
            'log_level': last_log_level,
            'group_id': last_group_id,
            'user_id': last_user_id,
            'message': line
        })
    
    return pd.DataFrame(parsed_logs)

# Function to filter logs by group
def filter_logs_by_group(df, group_id=None):
    if group_id is not None:
        return df[df['group_id'] == str(group_id)]
    else:
        # Group the data and return a dictionary
        grouped = defaultdict(list)
        for _, row in df.iterrows():
            if row['group_id'] is not None:
                grouped[row['group_id']].append(row)
        
        return {group: pd.DataFrame(logs) for group, logs in grouped.items()}

# Function to filter logs by user
def filter_logs_by_user(df, user_id=None):
    if user_id is not None:
        return df[df['user_id'] == str(user_id)]
    else:
        # Group the data and return a dictionary
        grouped = defaultdict(list)
        for _, row in df.iterrows():
            if row['user_id'] is not None:
                grouped[row['user_id']].append(row)
        
        return {user: pd.DataFrame(logs) for user, logs in grouped.items()}

# Function to extract all group members from the logs
def extract_group_members(df):
    group_members = {}
    
    # Find lines containing group user ids info
    user_id_lines = df[df['message'].str.contains('Group user ids:', na=False)]
    
    for _, row in user_id_lines.iterrows():
        if row['group_id'] is not None:
            group_id = row['group_id']
            # Extract the user IDs using regex
            match = re.search(r'Group user ids: \[(.*?)\]', row['message'])
            if match:
                user_ids_str = match.group(1)
                user_ids = [id.strip() for id in user_ids_str.split(',')]
                group_members[group_id] = user_ids
    
    return group_members

# Main function to process the log file
def process_log_file(log_content):
    # Parse the log content
    df = parse_log_file(log_content)
    
    # Extract group members
    group_members = extract_group_members(df)
    
    # Get unique groups and users
    unique_groups = df['group_id'].dropna().unique()
    unique_users = df['user_id'].dropna().unique()
    
    # Filter logs by groups
    group_logs = filter_logs_by_group(df)
    
    # Filter logs by users
    user_logs = filter_logs_by_user(df)
    
    return {
        'full_logs': df,
        'group_members': group_members,
        'unique_groups': unique_groups,
        'unique_users': unique_users,
        'group_logs': group_logs,
        'user_logs': user_logs
    }

# Example usage in a Jupyter notebook
# If running in a notebook with an uploaded file:
from IPython.display import display

# First, read the log file 
with open('app_log_bf_data_4.txt', 'r') as f:
    log_content = f.read()

# Process the log file
results = process_log_file(log_content)

# Print summary
print(f"Found {len(results['unique_groups'])} unique groups: {', '.join(results['unique_groups'])}")
print(f"Found {len(results['unique_users'])} unique users: {', '.join(results['unique_users'])}")
print("\nGroup members:")
for group, members in results['group_members'].items():
    print(f"Group {group}: {', '.join(members)}")

# Function to create filtered log files for each group
def save_group_logs(results, output_dir='.'):
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    for group_id, logs_df in results['group_logs'].items():
        # Sort logs by timestamp
        logs_df = logs_df.sort_values('timestamp')
        
        output_file = f"{output_dir}/group_{group_id}_logs.txt"
        with open(output_file, 'w') as f:
            for _, row in logs_df.iterrows():
                f.write(f"{row['message']}\n")
        print(f"Saved logs for Group {group_id} to {output_file}")

# Function to create filtered log files for each user
def save_user_logs(results, output_dir='.'):
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    for user_id, logs_df in results['user_logs'].items():
        # Sort logs by timestamp
        logs_df = logs_df.sort_values('timestamp')
        
        output_file = f"{output_dir}/user_{user_id}_logs.txt"
        with open(output_file, 'w') as f:
            for _, row in logs_df.iterrows():
                f.write(f"{row['message']}\n")
        print(f"Saved logs for User {user_id} to {output_file}")

# Function to save all logs for a specific group and its users in one file
def save_group_and_users_logs(results, group_id, output_dir='.'):
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    if group_id not in results['group_members']:
        print(f"Group {group_id} not found in logs")
        return
    
    # Get the logs for the specific group
    group_logs = results['group_logs'].get(group_id, pd.DataFrame())
    
    # Get the logs for all users in this group
    user_ids = results['group_members'][group_id]
    
    # Create a combined dataframe for all logs related to this group and its users
    all_logs = group_logs.copy()
    
    # Add logs for each user in the group
    for user_id in user_ids:
        if user_id in results['user_logs']:
            user_logs = results['user_logs'][user_id]
            all_logs = pd.concat([all_logs, user_logs])
    
    # Remove duplicates and sort by timestamp
    all_logs = all_logs.drop_duplicates().sort_values('timestamp')
    
    # Save the combined logs
    output_file = f"{output_dir}/group_{group_id}_with_users_logs.txt"
    with open(output_file, 'w') as f:
        for _, row in all_logs.iterrows():
            timestamp = row['timestamp'] or ''
            message = row['message']
            f.write(f"{message}\n")
    
    print(f"Saved combined logs for Group {group_id} and its users to {output_file}")

# Save filtered logs
group_id = '31'  # Based on the log sample
save_group_and_users_logs(results, group_id, 'filtered_logs')

# # Display sample of logs for a specific group
# group_id = '24'  # Based on the log sample
# if group_id in results['group_logs']:
#     print(f"\nSample logs for Group {group_id}:")
#     sample_logs = results['group_logs'][group_id].head(10)
#     for _, row in sample_logs.iterrows():
#         print(row['message'])

# Display user activity timeline
def analyze_user_activity(results, user_id):
    if user_id not in results['user_logs']:
        print(f"No logs found for User {user_id}")
        return
    
    user_df = results['user_logs'][user_id]
    
    # Extract pages from disconnect/reconnect logs
    disconnect_pages = []
    reconnect_pages = []
    
    for _, row in user_df.iterrows():
        if 'disconnect_pages' in row['message']:
            match = re.search(r"'disconnect_pages': \[(.*?)\]", row['message'])
            if match:
                pages_str = match.group(1)
                pages = re.findall(r"'(.*?)'", pages_str)
                disconnect_pages.extend(pages)
        
        if 'reconnect_pages' in row['message']:
            match = re.search(r"'reconnect_pages': \[(.*?)\]", row['message'])
            if match:
                pages_str = match.group(1)
                pages = re.findall(r"'(.*?)'", pages_str)
                reconnect_pages.extend(pages)
    
    # Create timeline of pages visited
    timeline = []
    for page in reconnect_pages:
        if page not in timeline:
            timeline.append(page)
    
    print(f"\nUser {user_id} Activity Timeline:")
    for i, page in enumerate(timeline):
        print(f"{i+1}. {page}")
    
    # Count disconnects/reconnects
    print(f"\nTotal disconnects: {len(disconnect_pages)}")
    print(f"Total reconnects: {len(reconnect_pages)}")

# # Analyze a specific user's activity
# user_id = '89'  # Based on the log sample
# analyze_user_activity(results, user_id)

Found 29 unique groups: 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 6, 32, 33, 34
Found 108 unique users: 18, 17, 16, 21, 20, 19, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68, 67, 69, 70, 71, 72, 75, 76, 77, 78, 79, 80, 81, 82, 83, 85, 84, 86, 87, 88, 89, 90, 91, 92, 13, 73, 74, 4, 93, 94, 95, 96, 97, 98, 99, 101, 100, 102, 103, 105, 104, 106, 107, 108, 109, 110, 111, 113, 114, 116, 117, 112, 115, 118, 119, 120, 121

Group members:
Group 7: 18, 17, 16
Group 8: 38, 39, 40
Group 9: 41, 42, 43
Group 12: 31, 32, 33
Group 13: 34, 35, 36
Group 14: 37, 38, 39
Group 15: 47, 55, 56
Group 16: 57, 58, 59
Group 17: 60, 61, 62
Group 18: 64, 72, 73
Group 19: 52, 53, 54
Group 20: 55, 56, 57
Group 21: 58, 59, 60
Group 24: 68, 67, 69
Group 25: 92, 93, 94
Group 26: 95, 96, 97
Group 27: 99, 100, 98
Group 28: 101, 102

In [ ]:
# parse activtity

import json
import pandas as pd
import sqlite3
import ast
import dill as pickle

# load sql database

data_path = '../data/full_study_data.db'
conn = sqlite3.connect(data_path)
all_users_df = pd.read_sql(f"SELECT * FROM {'user'}", conn)

user_id = '108_2'

# Filter the DataFrame for the specific user_id
user_df = all_users_df[all_users_df['id'] == user_id]

print(user_df['last_activity'])

pickled_data = user_df['last_activity'].values[0]
json_str = pickle.loads(pickled_data)


##################################################

# Load the JSON data
events = json.loads(json_str)

# Convert to DataFrame
df = pd.DataFrame(events)

# Ensure all expected keys exist as columns
for col in ['t', 'e', 'u', 'k', 'm', 'tg']:
    if col not in df.columns:
        df[col] = None

# Save to CSV
df.to_csv(f"events_log_{user_id}.csv", index=False)

print(f"Saved to events_log_{user_id}.csv")


char_count = len(json_str)
char_count



4    b'\x80\x05\x95\xcay\x00\x00\x00\x00\x00\x00X\x...
Name: last_activity, dtype: object
Saved to events_log_65_2.csv


31171

In [45]:
# Parse the demonstration generation log data
import dill as pickle
import pandas as pd
import json
import sys, os


print(os.getcwd())


sys.path.append(os.path.join(os.getcwd(), 'app/group_teaching/codes'))

print((os.path.join(os.getcwd(), 'app/group_teaching/codes')))

data_path = '../2025-04-30_group_38/BEC_summary.pickle'

demo_gen_log = pickle.load(open(data_path, 'rb'))

# convert to DataFrame
df = pd.DataFrame(demo_gen_log)

# Ensure all expected keys exist as columns
df.to_excel('../debug/BEC_summary.xlsx', index=False)



/home/sureshkj/Projects/flask_closed_loop_teaching_local_unstaged/simple_game_test
/home/sureshkj/Projects/flask_closed_loop_teaching_local_unstaged/simple_game_test/app/group_teaching/codes


ModuleNotFoundError: No module named 'teams'